# Chapter 5 Source Code Notebook

This notebook builds the source code for Chapter 5, **Planning, Search, and Decision Loops**.

The code builds a constrained planner that generates candidate paths, scores them, selects a route, and handles rollback for SupportOps AI.

Before running the notebook, install any required packages and set the required API keys in your environment where model powered examples are used.


## Step 1: Candidate Plans

### `planner/candidates.py`

Planning begins by defining what a candidate resolution path looks like. Each option carries the information needed for later scoring, selection, and rollback.

In [ ]:
# planner/candidates.py
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class PlanStep:
    action:        str          # tool name from ToolRegistry
    inputs:        dict
    required_evidence: list[str] = field(default_factory=list)
    # session fields that must be set before this step can execute
    reversible:    bool = True
    description:   str = ""

@dataclass
class CandidatePlan:
    plan_id:       str
    hypothesis:    str          # one-sentence rationale
    steps:         list[PlanStep]
    # Scoring components - filled in by ScoringFunction
    evidence_score:  float = 0.0   # 0.0 - 1.0
    cost_score:      float = 0.0   # 0.0 - 1.0 (1.0 = cheapest)
    risk_score:      float = 0.0   # 0.0 - 1.0 (1.0 = lowest risk)
    total_score:     float = 0.0
    executable:      bool = True   # False if evidence check fails
    blocked_reason:  str = ""


## Step 2: Candidate Generation

### `planner/generator.py`

Once the plan structure exists, the generator creates multiple possible paths for the same ticket. This gives the system alternatives to compare before committing to one route.

In [ ]:
# planner/generator.py
import json
import anthropic
from ..session import Session
from ..tracker import SystemTrace, tracked_call
from ..config import SystemConfig, DEFAULT_CONFIG
from .candidates import CandidatePlan, PlanStep

GENERATION_PROMPT = """
You are a support resolution planner. Generate {n} distinct candidate
resolution plans for the following ticket.

Ticket: {ticket}
Current session state: {session_summary}
Available tools: {tools}

For each plan return a JSON object:
{
  "plan_id": "plan_A",
  "hypothesis": "<one sentence explaining why this plan should work>",
  "steps": [
    {
      "action": "<tool_name>",
      "inputs": {{...}},
      "required_evidence": ["<session_field_that_must_exist>"],
      "reversible": true,
      "description": "<what this step does>"
    }
  ]
}

Return a JSON array of {n} plans. No preamble."""

def generate_candidates(
    client: anthropic.Anthropic,
    session: Session,
    available_tools: list[str],
    trace: SystemTrace,
    config: SystemConfig = DEFAULT_CONFIG,
    n: int = 3
) -> list[CandidatePlan]:
    session_summary = (
        f"routing={session.routing_category}, "
        f"classification={session.classification.category.value if session.classification else 'none'}, "
        f"escalated={session.escalated}, "
        f"steps_taken={len(session.actions_taken)}"
    )
    prompt = GENERATION_PROMPT.format(
        n=n,
        ticket=session.raw_input[:500],
        session_summary=session_summary,
        tools=", ".join(available_tools)
    )
    response = tracked_call(
        client, "plan_generate", trace,
        model=config.smart_model,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.content[0].text.strip()
    plans_data = json.loads(raw)
    candidates = []
    for d in plans_data[:n]:   # never more than n regardless of model output
        steps = [PlanStep(**s) for s in d.get("steps", [])]
        candidates.append(CandidatePlan(
            plan_id=d["plan_id"],
            hypothesis=d["hypothesis"],
            steps=steps
        ))
    return candidates


## Step 3: The Scoring Function

### `planner/scoring.py`

The scoring function then evaluates each candidate against evidence quality, cost, risk, and expected value. The planner can now compare options using explicit criteria instead of intuition hidden in a prompt.

In [ ]:
# planner/scoring.py
from dataclasses import dataclass
from ..session import Session
from .candidates import CandidatePlan
from ..agent.tools.contracts import ActionRisk

@dataclass
class ScoringWeights:
    evidence: float = 0.40   # how well the plan fits available evidence
    cost:     float = 0.25   # prefer cheaper plans
    risk:     float = 0.35   # prefer lower-risk plans

    def __post_init__(self):
        total = self.evidence + self.cost + self.risk
        assert abs(total - 1.0) < 0.001, f"Weights must sum to 1.0, got {total}"

# Cost estimate per tool call type
TOOL_COST_ESTIMATE = {
    "classify_ticket":  0.001,
    "check_escalation": 0.000,
    "draft_response":   0.004,
    "send_response":    0.000,
    "search_knowledge": 0.002,
    "get_history":      0.000,
}

TOOL_RISK = {
    "classify_ticket":  0.0,    # read, low risk
    "check_escalation": 0.0,
    "search_knowledge": 0.0,
    "get_history":      0.0,
    "draft_response":   0.2,    # write, medium risk
    "send_response":    0.8,    # side effect, high risk
}

def score_evidence(plan: CandidatePlan, session: Session) -> float:
    """
    Check that required evidence fields exist in session.
    Returns fraction of required fields that are populated.
    Also marks plan as not executable if any required field is missing.
    """
    all_required = []
    for step in plan.steps:
        all_required.extend(step.required_evidence)
    if not all_required:
        return 1.0   # no evidence requirements = full score
    session_dict = {
        "classification": session.classification,
        "routing_category": session.routing_category,
        "draft_response": session.draft_response,
        "retrieved_context": session.retrieved_context or None,
        "escalated": session.escalated
    }
    populated = [f for f in all_required if session_dict.get(f)]
    score = len(populated) / len(all_required)
    if score < 1.0:
        plan.executable = False
        missing = [f for f in all_required if not session_dict.get(f)]
        plan.blocked_reason = f"Missing required fields: {missing}"
    return score

def score_cost(plan: CandidatePlan) -> float:
    """Lower cost = higher score. Normalize to [0, 1]."""
    total = sum(TOOL_COST_ESTIMATE.get(s.action, 0.003)
               for s in plan.steps)
    # Cap at 0.02 (5 moderate calls); below that, scale linearly
    return max(0.0, 1.0 - (total / 0.02))

def score_risk(plan: CandidatePlan) -> float:
    """Lower risk = higher score."""
    if not plan.steps:
        return 1.0
    max_risk = max(TOOL_RISK.get(s.action, 0.3) for s in plan.steps)
    return 1.0 - max_risk

def score_plan(plan: CandidatePlan, session: Session,
               weights: ScoringWeights = None) -> CandidatePlan:
    if weights is None:
        weights = ScoringWeights()
    plan.evidence_score = score_evidence(plan, session)
    plan.cost_score     = score_cost(plan)
    plan.risk_score     = score_risk(plan)
    plan.total_score = (
        weights.evidence * plan.evidence_score +
        weights.cost     * plan.cost_score +
        weights.risk     * plan.risk_score
    )
    return plan


## Step 4: The Rollback Manager

### `planner/rollback.py`

Because selected plans can still fail, rollback support records what changed and how to recover. This keeps planning connected to safe execution rather than stopping at selection.

In [ ]:
# planner/rollback.py
import copy
from dataclasses import dataclass
from ..session import Session

@dataclass
class Checkpoint:
    plan_id:        str
    step_number:    int
    session_snapshot: dict   # shallow copy of key session fields

class RollbackManager:
    def __init__(self):
        self._checkpoints: list[Checkpoint] = []

    def save(self, plan_id: str, step_number: int,
              session: Session) -> None:
        snapshot = {
            "status":          session.status,
            "classification":  session.classification,
            "draft_response":  session.draft_response,
            "escalated":       session.escalated,
            "escalation_reason": session.escalation_reason,
            "actions_taken":   copy.copy(session.actions_taken)
        }
        self._checkpoints.append(Checkpoint(
            plan_id=plan_id,
            step_number=step_number,
            session_snapshot=snapshot
        ))

    def rollback(self, session: Session, plan_id: str) -> Session:
        """Restore session to the state before plan_id began."""
        target = next(
            (c for c in reversed(self._checkpoints)
             if c.plan_id == plan_id and c.step_number == 0),
            None
        )
        if target is None:
            return session   # no checkpoint found, return as-is
        snap = target.session_snapshot
        session.status           = snap["status"]
        session.classification   = snap["classification"]
        session.draft_response   = snap["draft_response"]
        session.escalated        = snap["escalated"]
        session.escalation_reason = snap["escalation_reason"]
        session.actions_taken    = snap["actions_taken"]
        return session


## Step 5: The Planner

### `planner/planner.py`

The planner combines generation, scoring, search depth, selection, and rollback awareness into one controlled decision loop. It chooses the strongest path while staying within configured limits.

In [ ]:
# planner/planner.py
from dataclasses import dataclass
from typing import Optional
import anthropic
from ..session import Session, SessionStatus
from ..tracker import SystemTrace
from ..reasoning.logger import StepLogger
from ..reasoning.budget import BudgetEnforcer
from ..config import SystemConfig, DEFAULT_CONFIG
from ..agent.tools.registry import ToolRegistry
from ..agent.controller import AgentController
from .generator import generate_candidates
from .scoring import score_plan, ScoringWeights
from .rollback import RollbackManager
from .candidates import CandidatePlan

@dataclass
class PlannerConfig:
    max_candidates:      int   = 3
    min_score_threshold: float = 0.35   # block if best score < this
    min_margin:          float = 0.10   # flag near-ties for review
    max_rollbacks:       int   = 2
    weights:             ScoringWeights = None

    def __post_init__(self):
        if self.weights is None:
            self.weights = ScoringWeights()

def _ambiguity_score(session: Session) -> float:
    """
    Estimate how ambiguous the ticket is.
    Returns 0.0 (obvious) to 1.0 (very ambiguous).
    Simple heuristic: more topics mentioned = higher ambiguity.
    """
    ticket = session.raw_input.lower()
    topics = [
        any(w in ticket for w in ["billing", "charge", "invoice", "payment"]),
        any(w in ticket for w in ["login", "access", "password", "account"]),
        any(w in ticket for w in ["error", "broken", "crash", "down", "fail"]),
        any(w in ticket for w in ["slow", "timeout", "latency", "performance"]),
    ]
    return sum(topics) / len(topics)

class Planner:
    def __init__(
        self,
        client:     anthropic.Anthropic,
        registry:   ToolRegistry,
        controller: AgentController,
        config:     PlannerConfig = None,
        sys_config: SystemConfig = DEFAULT_CONFIG
    ):
        self.client     = client
        self.registry   = registry
        self.controller = controller
        self.cfg        = config or PlannerConfig()
        self.sys_cfg    = sys_config

    def run(self, session: Session, trace: SystemTrace,
             budget: BudgetEnforcer,
             logger: StepLogger) -> Session:

        # ── Predict-vs-plan gate ─────────────────────────────────────
        ambiguity = _ambiguity_score(session)
        if ambiguity < 0.30:
            logger.log("planner_gate", {"ambiguity": ambiguity},
                {"decision": "fast_predict",
                 "reason": "low ambiguity - bypassing planner"},
                "success")
            return self.controller.run(session, trace, budget, logger)

        # ── Generate candidates ──────────────────────────────────────
        budget.tick()
        tool_names = [s.name for s in self.registry.all_specs()]
        candidates = generate_candidates(
            self.client, session, tool_names, trace, self.sys_cfg,
            n=self.cfg.max_candidates
        )
        logger.log(
            "generate_candidates",
            {"ambiguity": ambiguity},
            {"count": len(candidates),
             "plans": [c.plan_id for c in candidates]},
            "success"
        )

        # ── Score candidates ─────────────────────────────────────────
        for c in candidates:
            score_plan(c, session, self.cfg.weights)

        # Filter to executable plans only
        executable = [c for c in candidates if c.executable]
        if not executable:
            logger.log("score_candidates", {},
                {"result": "all_plans_blocked",
                 "reasons": [c.blocked_reason for c in candidates]},
                "failed",
                notes="No executable plan - escalating")
            session.escalated = True
            session.escalation_reason = "no_executable_plan"
            session.status = SessionStatus.ESCALATED
            return session

        executable.sort(key=lambda c: c.total_score, reverse=True)
        best = executable[0]

        # Score threshold check
        if best.total_score < self.cfg.min_score_threshold:
            logger.log("score_candidates",
                {"best_score": best.total_score},
                {"result": "below_threshold",
                 "threshold": self.cfg.min_score_threshold},
                "failed",
                notes="Best plan below minimum score - escalating")
            session.escalated = True
            session.escalation_reason = "plan_score_below_threshold"
            session.status = SessionStatus.ESCALATED
            return session

        # Log scores for all candidates
        score_summary = {
            c.plan_id: {
                "total": round(c.total_score, 3),
                "evidence": round(c.evidence_score, 3),
                "cost": round(c.cost_score, 3),
                "risk": round(c.risk_score, 3),
                "hypothesis": c.hypothesis
            } for c in executable
        }
        logger.log("score_candidates",
            {}, {"scores": score_summary, "selected": best.plan_id},
            "success")

        # ── Execute with rollback ─────────────────────────────────────
        rollback = RollbackManager()
        remaining = executable[:self.cfg.max_rollbacks + 1]

        for plan in remaining:
            rollback.save(plan.plan_id, 0, session)
            try:
                session = self._execute_plan(
                    plan, session, trace, budget, logger
                )
                if session.status == SessionStatus.RESOLVED:
                    logger.log("plan_success",
                        {"plan_id": plan.plan_id}, {}, "success")
                    return session
                # Plan didn't resolve - try next
                session = rollback.rollback(session, plan.plan_id)
                logger.log("plan_rollback",
                    {"plan_id": plan.plan_id},
                    {"reason": "plan_did_not_resolve"},
                    "failed")
            except Exception as e:
                session = rollback.rollback(session, plan.plan_id)
                logger.log("plan_rollback",
                    {"plan_id": plan.plan_id},
                    {"reason": str(e)},
                    "failed")

        # All plans exhausted
        session.escalated = True
        session.escalation_reason = "all_plans_exhausted"
        session.status = SessionStatus.ESCALATED
        return session

    def _execute_plan(self, plan: CandidatePlan, session: Session,
                       trace: SystemTrace, budget: BudgetEnforcer,
                       logger: StepLogger) -> Session:
        """Execute a single candidate plan step by step."""
        logger.log("execute_plan",
            {"plan_id": plan.plan_id, "steps": len(plan.steps)},
            {"hypothesis": plan.hypothesis},
            "success")
        for step in plan.steps:
            bstatus = budget.tick()
            if bstatus.exceeded:
                raise RuntimeError("Budget exceeded during plan execution")
            result = self.registry.execute(step.action, step.inputs)
            if not result.success:
                raise RuntimeError(
                    f"Step '{step.action}' failed: {result.error}"
                )
            # Reflect results into session
            if step.action == "classify_ticket" and result.success:
                session.classification = result.data
            elif step.action == "draft_response" and result.success:
                session.draft_response = result.data
            logger.log(
                f"plan_step:{step.action}",
                step.inputs,
                {"success": result.success,
                 "preview": str(result.data)[:80]},
                "success" if result.success else "failed"
            )
        if session.draft_response:
            session.status = SessionStatus.RESOLVED
        return session


## Step 6: Running the Planner

### `planner/run_planner.py`

The final cell runs the planner against a sample task and shows the selected path. It demonstrates how candidate generation, scoring, and selection work together.

In [ ]:
# planner/run_planner.py
import os
import anthropic
from supportops_ai.session import Session
from supportops_ai.tracker import SystemTrace
from supportops_ai.reasoning.budget import BudgetEnforcer
from supportops_ai.reasoning.logger import StepLogger
from supportops_ai.config import DEFAULT_CONFIG
from supportops_ai.agent.run_agent import registry, policy, controller
from supportops_ai.planner.planner import Planner, PlannerConfig
from supportops_ai.planner.scoring import ScoringWeights

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
config = DEFAULT_CONFIG

planner = Planner(
    client=client,
    registry=registry,
    controller=controller,
    config=PlannerConfig(
        max_candidates=3,
        min_score_threshold=0.35,
        min_margin=0.10,
        max_rollbacks=2,
        weights=ScoringWeights(evidence=0.40, cost=0.25, risk=0.35)
    ),
    sys_config=config
)

def process_with_planner(ticket_id: str,
                          raw_input: str) -> tuple[Session, str]:
    trace   = SystemTrace(ticket_id)
    session = Session(ticket_id=ticket_id, raw_input=raw_input)
    budget  = BudgetEnforcer(max_seconds=120.0, max_steps=12)
    logger  = StepLogger(session)
    session = planner.run(session, trace, budget, logger)
    return session, logger.trace()

if __name__ == "__main__":
    # Ambiguous ticket - should trigger planning
    ambiguous = (
        "I was charged twice and now my account shows as suspended. "
        "The API is also returning errors which is blocking my team. "
        "I need all of these resolved today."
    )
    session, trace = process_with_planner("TKT-005", ambiguous)
    print(trace)
    print(f"Status: {session.status.value}")
    print(f"Escalated: {session.escalated}")

    print("\n" + "="*50 + "\n")

    # Clear ticket - should bypass planning
    clear = "How do I reset my password?"
    session2, trace2 = process_with_planner("TKT-006", clear)
    print(trace2)
    print(f"Status: {session2.status.value}")
